In [9]:
from langgraph.graph import StateGraph,START,END
from dotenv import load_dotenv
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI


load_dotenv()


model = ChatGoogleGenerativeAI(model="gemini-3.8-flash")

#create a State
class LLMState(TypedDict):
    question:str
    answer:str


def llm_qa(state:LLMState)->LLMState:
    #extract question from State
    question=state['question']

    #form prompt
    prompt=f'Answer the following question{question}'

     #ask that question to LLM
    answer=model.invoke(prompt).content

    #update the answer in state
    state['answer']=answer

    return state


#create Graph
graph=StateGraph(LLMState)
#add nodes
graph.add_node('llm_qa',llm_qa)

#add edges
graph.add_edge(START,'llm_qa')
graph.add_edge('llm_qa',END)

#compile graph
workflow=graph.compile()
#execute graph
initial_state={'question':'What is Langgraph'}
output=workflow.invoke(initial_state)
#Show Grapgh
from IPython.display import Image
Image(workflow.get_graph().draw_mermaid_png())

print(output)

print(output['answer'])

{'question': 'What is Langgraph', 'answer': [{'type': 'text', 'text': '**LangGraph** is an open-source framework developed by the team behind LangChain. It is designed to build **robust, stateful, multi-actor (or multi-agent) applications** using Large Language Models (LLMs). \n\nWhile standard LangChain is great for linear, step-by-step sequences, LangGraph is built specifically for **cyclical workflows**—where an AI agent needs to loop back, reflect, critique, and retry tasks until a goal is achieved.\n\n---\n\n### The Problem LangGraph Solves\nTraditional LLM chains are **DAGs** (Directed Acyclic Graphs). This means information flows in only one direction: \n> *Step A $\\rightarrow$ Step B $\\rightarrow$ Step C $\\rightarrow$ Output.*\n\nHowever, real-world "Agentic" behavior requires loops:\n> *Plan $\\rightarrow$ Execute $\\rightarrow$ Review output $\\rightarrow$ **Did it fail? Loop back to Execute** $\\rightarrow$ Success $\\rightarrow$ Output.*\n\nLangGraph introduces the abili